In [ ]:
import io
import json
import math
from pathlib import Path

import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from PIL import Image, ImageOps, ImageCms, ImageFile

try:
    import pillow_heif
    pillow_heif.register_heif_opener()
except ImportError:
    pass

try:
    import pillow_avif
except ImportError:
    pass

ImageFile.LOAD_TRUNCATED_IMAGES = False
ROOT = Path.cwd()
ARTIFACTS = ROOT / "model_artifacts"
CONFIG = json.loads((ARTIFACTS / "inference_config.json").read_text(encoding="utf-8"))
MODEL_PATH = ARTIFACTS / CONFIG["model_file"]
model = tf.keras.models.load_model(str(MODEL_PATH), compile=False)
SIZE = int(CONFIG["image_size"])
THRESHOLD = float(CONFIG["decision_threshold"])
print(f"Loaded model: {MODEL_PATH}")
print(f"Input {SIZE}x{SIZE} RGB | Threshold {THRESHOLD:.4f}")
print(f"TensorFlow {tf.__version__} | GPU {tf.config.list_physical_devices('GPU')}")

In [ ]:
def clean_path(raw):
    raw = raw.strip()
    raw = raw.strip().strip(chr(34) + chr(39)).strip()
    raw = raw.replace("\\ ", " ")
    return Path(raw).expanduser().resolve()

def decode_rgb(path):
    with Image.open(path) as original:
        try:
            original.draft("RGB", (900, 900))
        except Exception:
            pass
        original.seek(0)
        img = ImageOps.exif_transpose(original)
        profile = img.info.get("icc_profile")
        if img.mode not in ("RGB", "RGBA"):
            img = img.convert("RGBA" if "A" in img.getbands() else "RGB")
        if profile and img.mode == "RGB":
            try:
                img = ImageCms.profileToProfile(img, ImageCms.ImageCmsProfile(io.BytesIO(profile)), ImageCms.createProfile("sRGB"), outputMode="RGB")
            except Exception:
                pass
        if img.mode == "RGBA":
            img = Image.alpha_composite(Image.new("RGBA", img.size, (255,255,255,255)), img).convert("RGB")
        return img.copy()

def crop_variant(img, variant):
    width, height = img.size
    side = min(width, height)
    if variant == "corner_tl":
        left, top = 0, 0
    elif variant == "corner_br":
        left, top = width - side, height - side
    else:
        left, top = (width - side) // 2, (height - side) // 2
    return img.crop((left, top, left + side, top + side)).resize((SIZE, SIZE), Image.Resampling.LANCZOS)

def get_predictions(img):
    variants = ["center", "corner_tl", "corner_br"]
    crops = [crop_variant(img, k) for k in variants]
    batch = np.stack([np.asarray(crop, dtype=np.float32) / 255.0 for crop in crops], axis=0)
    flipped = batch[:, :, ::-1, :].copy()
    augmented = np.concatenate([batch, flipped], axis=0)
    predictions = model.predict(augmented, batch_size=6, verbose=0).reshape(-1)
    score_center = float(predictions[0])
    score_tta = float(predictions.mean())
    disagreement = float(np.std(predictions))
    return score_center, score_tta, disagreement, crops, predictions

print("Supported formats depend on the installed Pillow plugins, not filename extensions")

In [ ]:
raw = input("Enter the image path: ")
path = clean_path(raw)
if not path.is_file():
    raise FileNotFoundError(f"No image exists at: {path}")
try:
    image = decode_rgb(path)
except Exception as exc:
    raise ValueError(f"This file cannot be decoded as an image: {path}") from exc

center_score, tta_score, variability, crops, individual_scores = get_predictions(image)
label = "AI-GENERATED" if tta_score >= THRESHOLD else "REAL / CAMERA-SOURCED"
color = "#FFAD77" if tta_score >= THRESHOLD else "#56E0D4"
print(f"Prediction: {label}")
print(f"AI score: {tta_score:.4f} | threshold: {THRESHOLD:.4f} | crop variation: {variability:.4f}")
print("A score is a model output, not a scientifically calibrated probability or proof of origin")

plt.rcParams.update({"font.family": "DejaVu Sans", "figure.facecolor": "#080F1C", "text.color": "#EDF5FF", "axes.facecolor": "#111D31", "savefig.facecolor": "#080F1C"})
fig = plt.figure(figsize=(15.6, 7.8), dpi=135)
gs = GridSpec(2, 2, figure=fig, width_ratios=[1.2, 0.8], height_ratios=[1, 0.32], wspace=0.12, hspace=0.22)
ax_image = fig.add_subplot(gs[:, 0])
ax_meter = fig.add_subplot(gs[0, 1])
ax_detail = fig.add_subplot(gs[1, 1])
ax_image.imshow(image)
ax_image.set_axis_off()
ax_image.set_title("IMAGE UNDER EXAMINATION", loc="left", fontsize=14, weight="bold", pad=15)
ax_meter.set_xlim(0, 1)
ax_meter.set_ylim(0, 1)
ax_meter.axis("off")
ax_meter.text(0.02, 0.91, "AEGIS  /  CNN FORENSICS", size=12, weight="bold", color="#91A4C2", transform=ax_meter.transAxes)
ax_meter.text(0.02, 0.68, label, size=20, weight="bold", color=color, transform=ax_meter.transAxes)
ax_meter.text(0.02, 0.48, f"{tta_score * 100:.2f}%", size=45, weight="bold", color="#EFF6FF", transform=ax_meter.transAxes)
ax_meter.text(0.02, 0.38, "AI-class score (not calibrated confidence)", size=10, color="#91A4C2", transform=ax_meter.transAxes)
ax_meter.barh(0.25, 1, left=0.02, height=0.057, color="#2D415C")
ax_meter.barh(0.25, tta_score * 0.96, left=0.02, height=0.057, color=color)
ax_meter.plot([0.02 + 0.96*THRESHOLD]*2, [0.20, 0.30], color="#EEF5FF", lw=2)
ax_meter.text(0.02, 0.11, f"Decision boundary: {THRESHOLD:.3f}   •   Crop variation: {variability:.4f}", color="#BED0E8", size=10, transform=ax_meter.transAxes)
ax_detail.axis("off")
ax_detail.text(0.02, 0.78, f"FILE   {path.name[:62]}", fontsize=10, weight="bold", color="#EEF5FF", transform=ax_detail.transAxes)
ax_detail.text(0.02, 0.47, f"SOURCE RESOLUTION   {image.width:,} × {image.height:,}   |   TTA VIEWS   {len(individual_scores)}", fontsize=9, color="#91A4C2", transform=ax_detail.transAxes)
ax_detail.text(0.02, 0.14, "Warning: generated images can be misclassified. Verify origin independently.", fontsize=8.4, color="#B9C7D9", transform=ax_detail.transAxes)
fig.suptitle("VISUAL AUTHENTICITY ANALYSIS", fontsize=19, weight="bold", x=0.02, y=0.985, ha="left")
plt.show()